In [86]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
url= "https://raw.githubusercontent.com/wonderakwei/telecom-customer-churn-prediction/refs/heads/main/churn-bigml-80.csv"
url1="https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/refs/heads/master/data/Telco-Customer-Churn.csv"
df=pd.read_csv(url1)
#print(df.shape)
#df.head()
#print(df.dtypes)
#df.tail()
#df.info()
#df.isna()



##Check for literal blank string spaces in the column


df["TotalCharges"]=pd.to_numeric(df['TotalCharges'], errors='coerce')

df['TotalCharges'] = df['TotalCharges'].fillna(0)
#df.info()

## Convert Churn column to binary integers
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

## Quick verification check
print(df['Churn'].value_counts())
# 1. Separate the features (X) and the target variable (y)
X = df.drop(columns=['Churn'])
y = df['Churn']
X=X.drop(columns="customerID")
#X.info()
## 2. Apply One-Hot Encoding to all categorical columns in X
# drop_first=True is an interview golden rule! It drops one redundant column per category 
# to prevent the "dummy variable trap" (multicollinearity) which breaks linear models.
X_encoded = pd.get_dummies(X, drop_first=True)

# 3. View the new shape and see how our data columns expanded
print(f"Original features shape: {X.shape}")
print(f"Encoded features shape: {X_encoded.shape}")
#print(X_encoded.head())
#X_encoded.head()
#X_encoded.info()

## TEST-TRAIN SLIT
x_train,x_test,y_train,y_test=train_test_split(X_encoded,y,test_size=0.2, random_state=42, stratify=y)
scaler=StandardScaler()

Xtrain_scaled=scaler.fit_transform(x_train)
Xtest_scaled=scaler.transform(x_test)
Xtrain_scaled=pd.DataFrame(Xtrain_scaled, columns=x_train.columns)
Xtest_scaled=pd.DataFrame(Xtest_scaled, columns=x_test.columns)
Xtrain_scaled.head(2)

print(Xtrain_scaled.isna().values.any())
# 1. Initialize the model with class_weight='balanced' to handle the imbalance
# random_state=42 ensures your weights initialize the exact same way every run
model = LogisticRegression( random_state=42, max_iter=1000)

# 2. Train (Fit) the model on your scaled training data
model.fit(Xtrain_scaled, y_train)

# 3. Use the trained model to make predictions on your scaled testing data
y_pred = model.predict(Xtest_scaled)

print("🎉 Model training complete! Let's look at the results:")
# 1. Print the standard accuracy score
print(f"Overall Accuracy Score: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")

# 2. Print the detailed Classification Report (Crucial for interviews!)
print("Detailed Classification Report:")
print(classification_report(y_test, y_pred))



Churn
0    5174
1    1869
Name: count, dtype: int64
Original features shape: (7043, 19)
Encoded features shape: (7043, 30)
False
🎉 Model training complete! Let's look at the results:
Overall Accuracy Score: 80.70%

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.57      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409

